# Reading and Writing Data

Load structured files carefully, preserve data types, and export analysis results.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Read CSV data with explicit parsing and missing-value options.
- Read and write Excel files when `openpyxl` is installed.
- Export a clean table and avoid common encoding, delimiter, and date issues.

Always inspect the raw file's delimiter, encoding, header row, and missing-value conventions. Do not assume the first read has interpreted every column correctly.

In [1]:
from io import StringIO
import pandas as pd

csv_text = """order_id,order_date,amount,region
1001,2025-01-02,35.50,North
1002,2025-01-03,,South
1003,2025-01-05,18.25,North
"""
orders = pd.read_csv(StringIO(csv_text), parse_dates=["order_date"])
display(orders)
print(orders.dtypes)

order_id               int64
order_date    datetime64[ns]
amount               float64
region                object
dtype: object


,order_id,order_date,amount,region
0,1001,2025-01-02,35.50,North
1,1002,2025-01-03,NaN,South
2,1003,2025-01-05,18.25,North


## Useful `read_csv` options
- `dtype=` can preserve identifiers with leading zeroes.
- `na_values=` tells pandas which strings mean missing.
- `usecols=` reduces memory and focuses the read.
- `chunksize=` returns an iterator for larger-than-memory files.
- `encoding="utf-8"` is a good starting point, but inspect the source if text appears corrupted.

In [2]:
csv_text = """postal_code;value;status
00120;4.2;valid
00121;NA;unknown
00122;5.1;valid
"""
parsed = pd.read_csv(StringIO(csv_text), sep=";", dtype={"postal_code": "string"}, na_values=["NA"])
display(parsed)
print(parsed.dtypes)

# Read selected columns only.
selected = pd.read_csv(StringIO(csv_text), sep=";", usecols=["postal_code", "value"])
display(selected)

postal_code    string[python]
value                 float64
status                 object
dtype: object


,postal_code,value,status
0,00120,4.2,valid
1,00121,NaN,unknown
2,00122,5.1,valid


,postal_code,value
0,120,4.2
1,121,NaN
2,122,5.1


## CSV and Excel export
Exporting a DataFrame does not preserve every metadata detail. CSV has no native type system, so parse dates and identifiers explicitly when reading it again.

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    csv_path = tmp / "clean_orders.csv"
    excel_path = tmp / "clean_orders.xlsx"
    orders.to_csv(csv_path, index=False, date_format="%Y-%m-%d")
    back = pd.read_csv(csv_path, parse_dates=["order_date"])
    display(back)
    try:
        orders.to_excel(excel_path, index=False)
        back_excel = pd.read_excel(excel_path, parse_dates=["order_date"])
        display(back_excel)
        print("Excel round-trip succeeded")
    except ImportError:
        print("Excel support requires openpyxl; install it with pip.")

Excel round-trip succeeded


,order_id,order_date,amount,region
0,1001,2025-01-02,35.50,North
1,1002,2025-01-03,NaN,South
2,1003,2025-01-05,18.25,North


,order_id,order_date,amount,region
0,1001,2025-01-02,35.50,North
1,1002,2025-01-03,NaN,South
2,1003,2025-01-05,18.25,North


## Large files: process in chunks
Chunk processing is helpful when a file is too large to load at once. The example uses an in-memory CSV stream, but the same pattern works with a file path.

In [4]:
large_like_csv = """category,amount
A,10
B,20
A,15
C,9
B,14
"""
partial_totals = {}
for chunk in pd.read_csv(StringIO(large_like_csv), chunksize=2):
    totals = chunk.groupby("category")["amount"].sum()
    for category, total in totals.items():
        partial_totals[category] = partial_totals.get(category, 0) + total
print(partial_totals)

{'A': 25, 'B': 34, 'C': 9}


## Practice
Read a semicolon-separated string, preserve an ID such as `0007`, parse the date, and export without writing the DataFrame index.

In [5]:
sample = "id;date;value\n0007;2025-06-01;3.4\n0008;2025-06-02;4.1"
out = pd.read_csv(StringIO(sample), sep=";", dtype={"id": "string"}, parse_dates=["date"])
display(out)
print(out.dtypes)

id       string[python]
date     datetime64[ns]
value           float64
dtype: object


,id,date,value
0,0007,2025-06-01,3.4
1,0008,2025-06-02,4.1
